In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Definition} $$

$$
(0, 1)^n \times \{0, 1\}^n \to \mathbb{R}, \quad L(\mathbf{p}, \mathbf{t}) = 
-\frac{1}{N} \sum_i t_i \, \ln(p_i) 
$$

$$ \sum_i p_i = 1, \quad \sum_i t_i = 1 $$

$$ \text{Definition} $$

$$ \frac{dL}{dp_i} = - \frac{1}{N} \, t_i \frac{1}{p_i} $$

$$ 
\frac{dL}{dp} = 
-\frac{1}{N} \, \mathbf{t} \oslash \mathbf{p} 
$$

$$ \text{Jacobian} $$

$$
J_L(\mathbf{p}) =
\nabla _L(\mathbf{p}) \top =
\begin{bmatrix}
    \frac{dL}{dp_1}, &
    \frac{dL}{dp_2},  &
    \cdots, &
    \frac{dL}{dp_n}
\end{bmatrix}
$$

$$ d\mathbf{L} = J_L(\mathbf{p}) \, d\mathbf{p} $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{d\mathbf{p}}, d\mathbf{p} \right \rangle =
\left \langle \frac{dF}{d\mathbf{y}}, d\mathbf{y} \right \rangle
$$

$$ \\[2em] $$

$$
\left \langle \frac{dF}{dL}, dL \right \rangle =
\left \langle \frac{dF}{dL}, J_L(\mathbf{p}) \, d\mathbf{p} \right \rangle =
\left \langle J_L(\mathbf{p}) ^\top \, \frac{dF}{dL}, d\mathbf{p} \right \rangle \implies
$$

$$ 
\frac{dF}{dp} = 
J_L(\mathbf{p})^\top \, \frac{dF}{dL} = 
\frac{dL}{dp} \odot \frac{dF}{dL} 
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import common # type: ignore
import softmax # type: ignore
from approx import approx # type: ignore

def _ce(p: tr.Tensor, t: tr.Tensor, reduction='mean') -> tr.Tensor:
    """
    Computes the cross-entropy loss between predictions `p` and targets `t` with `mean` reduction.
    Note: This function assumes that `p` has already been clamped to avoid log(0) issues.
    """

    if reduction != 'mean':
        assert False, f"Unsupported reduction: {reduction}"

    N = p.shape[0]
    return -1/N * (t * tr.log(p)).sum()


def ce(p: tr.Tensor, t: tr.Tensor, reduction='mean') -> tr.Tensor:
    """
    Computes the cross-entropy loss between predictions `p` and targets `t` with `mean` reduction.
    Note: This function clamps `p` to avoid log(0) issues.
    """

    assert tr.all((p >= -1.0) & (p <= +1.0))

    eps = 1e-7
    pc = p.clamp(eps, 1 - eps)
    return _ce(pc, t, reduction)


class CEFunction(autograd.Function):
    """Custom autograd function for the `cross-entropy`."""

    @staticmethod
    def forward(ctx, p, t):
        eps = 1e-7
        pc = p.clamp(eps, 1 - eps)
        ctx.save_for_backward(pc, t)
        return _ce(pc, t)

    @staticmethod
    def backward(ctx, dF_dL):
        (pc, t) = ctx.saved_tensors
        N = pc.shape[0]

        dL_dp = -(1 / N) * (t / pc)
        dF_dp = dL_dp * dF_dL

        return (dF_dp, None)
    
class CE(nn.Module):
    """Cross-entropy loss module."""

    def __init__(self, reduction='mean'):
        super().__init__()

        if reduction != "mean":
            assert False, f"Unsupported reduction: {reduction}"

    def forward(self, p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        return CEFunction.apply(p, t)

In [ ]:
# Unit tests

def test_gradcheck(samples=10):
    def fn(p, t):
        return CEFunction.apply(p, t)

    p = tr.rand(samples, 3, dtype=tr.float64, requires_grad=True)
    t = tr.randint(0, 5, (samples, 3)).float().requires_grad_(False)
    assert autograd.gradcheck(fn, (p, t), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    p = tr.rand(samples, 3, requires_grad=True)
    t = tr.randint(0, 5, (samples, 3)).float()

    p1 = p.clone().detach().requires_grad_(True)
    L1 = CE(reduction='mean')(softmax.Softmax()(p1), t)
    L1.backward()

    p2 = p.clone().detach().requires_grad_(True)
    L2 = nn.CrossEntropyLoss(reduction='mean')(p2, t)
    L2.backward()

    assert L1.item() == approx(L2.item()).log()
    assert p1.grad == approx(p2.grad)


if __name__ == "__main__":
    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)